# 04. From an observation attempt to a typed measurement

**Level:** Intermediate to advanced

**Prerequisites:** Lessons 020-030 and basic familiarity with measurement vocabulary.

**Canonical claim anchors:** `WDP-CH12-D01`, `WDP-CH12-P01`, `WDP-CH13-D01`, `WDP-CH13-N01`

## Learning objectives

- validate every rung of the observation ladder
- separate measurand, reported value, scale, calibration, and uncertainty
- keep WDP provenance distinct from metrological traceability

Work top-to-bottom. Predict each worked result before executing it, then use
the exercise to test whether you can separate the finite result from its scope
boundary.


In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
root = next((candidate for candidate in (start, *start.parents)
             if (candidate / 'pyproject.toml').is_file()
             and (candidate / 'code' / 'witnessed_difference').is_dir()), None)
if root is None:
    raise RuntimeError('could not locate the self-contained paper package')
sys.path.insert(0, str(root / 'code'))


## Idea: an attempt can exist without an accepted observation

An `ObservationReceipt` may record missing, rejected, refused, or accepted
stages. `classify_observation` promotes it to `AcceptedObservation` only when
there is a mark-capable interaction, a retained registration coupled to that
interaction, a matching replayable witness, and an evaluator with both
`Accept` status and accepting standing.

Measurement is a second typed layer. Acceptance alone does not supply a
measurand, result carrier, unit, calibration, uncertainty, provenance, or
traceability chain.


## Visual checkpoint

![Witness Ladder](../images/generated/witness_ladder.svg)

![Observer Types](../images/generated/observer_types.svg)

These files are deterministic generated aids. Their captions and the executed data, not visual resemblance alone, carry the intended lesson.


## Derivation

The worked packet reports terminal voltage under a declared loading condition.
Its result carrier is a display reading, not the measurand itself. A declared
map binds the retained display indication to that result carrier. The local
calibration correction is added to the exact raw value. Combined uncertainty
in this finite model is the stated uncertainty plus each declared link
contribution. That deliberately simple rule is package data, not a universal
metrology formula.


In [ ]:
from fractions import Fraction
from witnessed_difference.measurement import Calibration, CalibrationLink, IndicationValueMap, MeasurandSpecification, MetrologicalTraceability, NumericScale, Provenance, ReportedValue, measure
from witnessed_difference.notebook_support import display_json
from witnessed_difference.observation import Interaction, EvaluatorResult, EvaluatorStanding, EvaluatorStatus, ObservationReceipt, classify_observation
from witnessed_difference.witness import Registration, make_witness

registration = Registration('sensor', {'display': 12}, 0)
witness = make_witness(registration, {'display': 12}, 'E').value
attempt = ObservationReceipt(
    Interaction(('sensor', 'sample'), 'sensor', mark_capacity=True), registration, witness,
    EvaluatorResult('E', EvaluatorStatus.ACCEPT, EvaluatorStanding.ACCEPTING),
)
accepted = classify_observation(attempt)
measurement = measure(
    accepted.value,
    MeasurandSpecification('terminal voltage', 'sample', ('declared loading condition',)),
    ReportedValue('digital-display', 12),
    NumericScale('voltage-scale', 'V', Fraction(0), Fraction(20), Fraction(1, 2)),
    Calibration('cal-7', 'voltage-scale', 'E', Fraction(1, 2)),
    Fraction(1, 4),
    Provenance(('registration:0', 'reaccess:E')),
    MetrologicalTraceability(
        (CalibrationLink('cal-7', 'reference-1', 'certificate-X', Fraction(1, 10)),),
        'reference-1', 'external_standard', 'additive_upper_bound',
    ),
    IndicationValueMap('sensor-to-display', 'sensor', 'digital-display', 'display'),
)
print(display_json({'observation': accepted, 'measurement': measurement}))


## Read the result

The corrected result is `25/2 V` and the combined uncertainty is `7/20 V` in
this declared additive model. The packet also carries `AuthorityEffect: None`.
The chain's presence is an asserted, machine-checked shape; this notebook does
not independently authenticate certificate `X`.


In [ ]:
rejected_attempt = ObservationReceipt(
    attempt.interaction, registration, witness,
    EvaluatorResult('E', EvaluatorStatus.REJECT, EvaluatorStanding.ACCEPTING),
)
wrong_coupling = ObservationReceipt(
    Interaction(('other', 'sample'), 'other'), registration, witness,
    EvaluatorResult('E', EvaluatorStatus.ACCEPT, EvaluatorStanding.ACCEPTING),
)
print(display_json({'rejected_status': classify_observation(rejected_attempt), 'wrong_coupling': classify_observation(wrong_coupling)}))


## Exercise

Keep the accepted observation but replace metrological traceability with the
`Provenance` object. Predict the refusal code. Why is refusing this substitution
stronger than checking that both records are nonempty?


In [ ]:
confused = measure(
    accepted.value,
    MeasurandSpecification('terminal voltage', 'sample', ('declared loading condition',)),
    ReportedValue('digital-display', 12),
    NumericScale('voltage-scale', 'V', 0, 20, '1/2'),
    Calibration('cal-7', 'voltage-scale', 'E'),
    '1/4', Provenance(('registration:0',)), Provenance(('not-a-calibration-chain',)),
    IndicationValueMap('sensor-to-display', 'sensor', 'digital-display', 'display'),
)
print(display_json(confused))


## Scope check

The code validates a finite packet and declared chain shape. It does
not confer accreditation, establish physical truth, or authorize a decision.
**AuthorityEffect: None.**


## Continue the learning path

- Previous: [Three observer representations and explicit bridges](030_observer_representations.ipynb)
- Index: [Notebook curriculum](README.md)
- Next: [Witnessed duration and finite multiplicity](050_duration_and_finite_probability.ipynb)
